# HuBERT: Complete PyTorch Implementation & Audio Pipeline

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class ConvFeatureExtractionBlock(nn.Module):
    """
    Single 1D Convolution block with LayerNorm and GELU.
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride):
        super().__init__()
        
        self.conv = nn.Conv1d(
            in_channels, out_channels, 
            kernel_size=kernel_size, stride=stride, bias=False
        )

        self.layer_norm = nn.GroupNorm(num_groups=1, 
                num_channels=out_channels)
        self.activation = nn.GELU()

    def forward(self, x):
        # Input shape: (B, C_in, L_in)
        x = self.conv(x)
        x = self.layer_norm(x)
        x = self.activation(x)
        return x

class HuBERTFeatureEncoder(nn.Module):
    """
    7-layer 1D CNN waveform encoder downsampling 16kHz audio by 320x.
    """
    def __init__(self, embed_dim=512):
        super().__init__()
        conv_specs = [
            (1, 512, 10, 5),   # Layer 1: stride 5, kernel 10
            (512, 512, 3, 2),  # Layer 2: stride 2, kernel 3
            (512, 512, 3, 2),  # Layer 3: stride 2, kernel 3
            (512, 512, 3, 2),  # Layer 4: stride 2, kernel 3
            (512, 512, 3, 2),  # Layer 5: stride 2, kernel 3
            (512, 512, 2, 2),  # Layer 6: stride 2, kernel 2
            (512, 512, 2, 2),  # Layer 7: stride 2, kernel 2
        ]
        self.layers = nn.ModuleList([
            ConvFeatureExtractionBlock(in_ch, out_ch, k, s)
            for in_ch, out_ch, k, s in conv_specs
        ])

    def forward(self, raw_audio):
        # raw_audio: (B, Length_samples)
        x = raw_audio.unsqueeze(1)  # (B, 1, Length_samples)
        for layer in self.layers:
            x = layer(x)
        x = x.transpose(1, 2)  # (B, T_frames, 512)
        return x